# 05 — Gemma fine-tuned for Sinhala MCQs (train_data, tuned)

Fine-tunes Gemma (under 8B parameters) on the Sinhala past-paper questions in `data/train_data`, and tests on the **Dev Set**.

| Step | What happens |
|---|---|
| Data | `train.jsonl` (9,432 extracted questions, grades 6–13), `val.jsonl` (497), `test.jsonl` = the Dev Set (1,851) |
| Sinhala handling | every question/option is Unicode-normalised the same way (NFC, joiner kept for ශ්‍රී, ක්‍ය), options are shuffled into extra copies so the model must read them, lower grades (closest to the grade 6–8 test) can be given more weight, and the prompt language (English vs Sinhala instructions) is tuned |
| Hyperparameter search | short trials (subset, 1 epoch) over learning rate, LoRA rank and prompt language; scored on `val` |
| Final training | the best settings on all training data; the epoch with the best `val` accuracy is kept |
| Test | once, at the end, on the Dev Set: overall, per subject, per grade, vs the untrained model |

**Time on a Kaggle T4** (estimates; the progress bar shows the real speed): fp16 ≈ 5–6 h for everything; if Gemma has to fall back to fp32 (printed when the model loads) ≈ 3× longer, more than Kaggle's 12-hour limit. So:

* `TIME_BUDGET_HOURS` (default 11): training stops early when time runs short, **keeps the best epoch so far** and still runs the test, so a run always ends with a result.
* The steps can be split over **two Kaggle runs**: run 1 with `DO_SEARCH = True, DO_FINAL = False`; then copy the best settings it prints into `DEFAULT_HP` and do run 2 with `DO_SEARCH = False, DO_FINAL = True`.
* Faster: fewer `SEARCH_GRID` entries, `EPOCHS = 2`, or `AUG_SHUFFLES = 0` (halves training time).

**Before running:** same setup as notebook 04 — accept the Gemma licence on Hugging Face, add `HF_TOKEN` (and `GITHUB_TOKEN` for a private repo) as secrets, GPU on, internet on, and push `data/train_data/` to GitHub.

## 1. Environment

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU is plenty for 3B (Kaggle gives two)

GH_USER = "Imalsha-Ranasinghe"
REPO    = f"github.com/{GH_USER}/SinhalaMMLU-Shared-Task.git"
BRANCH  = "main"          # the branch to run; it must contain the latest src/ and notebooks

# Kaggle images also contain the google.colab package, so check Kaggle first, by its own env var.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB  = not IN_KAGGLE and "COLAB_RELEASE_TAG" in os.environ
print("Running on:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local / other")

def get_secret(name):
    # Read a secret from env vars, then Kaggle or Colab secrets
    if os.getenv(name):
        return os.getenv(name).strip()
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name).strip()
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name).strip()
    except Exception as e:
        print(f"[{name}] could not read secret: {type(e).__name__}: {e}")
    return None

clone_dir = Path.cwd() / "SinhalaMMLU-Shared-Task"
if (Path.cwd() / "src").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    ROOT = Path.cwd().parent
else:
    # Colab / Kaggle: always clone fresh so the code matches GitHub. Results live outside this folder.
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    gh_token = get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN found:", bool(gh_token))
    auth = f"{GH_USER}:{gh_token}@" if gh_token else ""
    r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, f"https://{auth}{REPO}", str(clone_dir)],
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        err = r.stderr.replace(gh_token, "***") if gh_token else r.stderr
        raise RuntimeError("git clone failed: " + err)
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", str(clone_dir), "remote", "set-url", "origin", f"https://{REPO}"])
    ROOT = clone_dir
    commit = subprocess.run(["git", "-C", str(clone_dir), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
    print(f"Cloned {BRANCH} @ {commit.strip()}")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

# Kaggle/Colab already ship most of these; install only what is missing or too old (fewer ways to fail).
import importlib, time
from packaging.version import Version
REQUIRED = {"transformers": "4.56", "peft": "0.13", "accelerate": "1.0", "bitsandbytes": "0.45"}
missing = []
for pkg, min_version in REQUIRED.items():
    try:
        if Version(importlib.import_module(pkg).__version__) < Version(min_version):
            missing.append(f"{pkg}>={min_version}")
    except Exception:
        missing.append(f"{pkg}>={min_version}")
if missing:
    print("installing:", missing)
    for attempt in (1, 2):                 # a second try covers temporary download errors
        pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
        if pip.returncode == 0:
            break
        time.sleep(10)
    if pip.returncode != 0:                # show pip's own error instead of a bare CalledProcessError
        print(pip.stdout[-3000:], pip.stderr[-3000:], sep="\n")
        raise RuntimeError("pip install failed - see pip's messages above")
print("packages ready:", {p: importlib.import_module(p).__version__ for p in REQUIRED})

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. On Colab: Runtime -> Change runtime type -> T4 GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Hugging Face login (Gemma 3 is a gated model)
from huggingface_hub import login

token = get_secret("HF_TOKEN")
if token:
    login(token=token)
    print("Logged in to Hugging Face")
else:
    raise RuntimeError("HF_TOKEN not found. Kaggle: Add-ons -> Secrets, add HF_TOKEN and tick its checkbox. "
                       "Colab: key icon -> add HF_TOKEN with Notebook access on.")

## 2. Configuration — edit this cell

In [ ]:
MODEL_ID     = "google/gemma-3-4b-it"     # any Gemma under 8B works; ungated copy: "unsloth/gemma-3-4b-it"
SHORT_NAME   = "gemma3-4b-sinhala"
MAX_PARAMS   = 8e9
LOAD_IN_4BIT = True
FORCE_FP32   = False          # set True if training shows loss = nan on a T4/P100
SEED         = 42
DATA_DIR     = ROOT / "data" / "train_data"
SCORER_TOKENS = 8000          # tokens per forward pass when scoring; lower if CUDA runs out of memory

# --- Sinhala data handling ---
AUG_SHUFFLES       = 1        # extra copies of each question with the options reordered (0 = off)
LOW_GRADE_BOOST    = 1.0      # extra weight for grades <= 9 (1.0 = off)
TEST_SUBJECT_BOOST = 1.0      # weight for the 14 subjects that appear in the test set (1.0 = off)

# --- What to run (split over two Kaggle runs if needed, see the top of the notebook) ---
DO_BASELINE       = True      # untrained model on val/test, for comparison
DO_SEARCH         = True      # hyperparameter search
DO_FINAL          = True      # final training + test
TIME_BUDGET_HOURS = 11        # Kaggle stops a run at 12 h; training stops early before that
NOTEBOOK_START    = __import__("time").time()

# --- Hyperparameter search: short trials, the best val accuracy wins ---
SEARCH_TRAIN_SIZE = 3000      # questions per trial (1 epoch)
SEARCH_GRID = [
    {"lang": "en", "lr": 1e-4, "r": 16},
    {"lang": "en", "lr": 2e-4, "r": 16},
    {"lang": "en", "lr": 1e-4, "r": 64},
    {"lang": "si", "lr": 1e-4, "r": 16},   # Sinhala instructions instead of English
]
DEFAULT_HP = {"lang": "en", "lr": 1e-4, "r": 16}   # used when no search results exist: paste the winner of run 1 here

# --- Final training ---
EPOCHS       = 3
BATCH_SIZE   = 8
GRAD_ACCUM   = 2              # effective batch = 16
MAX_LEN      = 512
LORA_DROPOUT = 0.05

# Persistent storage, so a cut-off session can resume instead of starting over.
if IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working")
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive/SinhalaMMLU-runs")
else:
    PERSIST_DIR = ROOT
RESULTS_DIR = PERSIST_DIR / "results" / SHORT_NAME
RUNS_DIR    = PERSIST_DIR / "outputs" / SHORT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def minutes_left(reserve=30):
    # Minutes left in the time budget, keeping `reserve` minutes for the final test.
    import time
    return TIME_BUDGET_HOURS * 60 - (time.time() - NOTEBOOK_START) / 60 - reserve

print("Results ->", RESULTS_DIR, "\nAdapters ->", RUNS_DIR)

## 3. Parameter budget check

In [ ]:
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM

with init_empty_weights():
    _m = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID))
n_params = sum(p.numel() for p in _m.parameters())   # tied embeddings counted once
del _m
print(f"{MODEL_ID}: {n_params / 1e9:.2f}B parameters (limit {MAX_PARAMS / 1e9:.0f}B)")
assert n_params < MAX_PARAMS

## 4. Data

In [ ]:
import pandas as pd
from src.data import load_jsonl, oversample

data = {s: load_jsonl(DATA_DIR / f"{s}.jsonl") for s in ["train", "val", "test"]}
for s, d in data.items():
    print(f"{s:5s}: {len(d):5d} questions, {d['subject'].nunique()} subjects")

test_subjects = set(data["test"]["subject"])
train_weighted = oversample(data["train"], subjects=test_subjects, subject_factor=TEST_SUBJECT_BOOST,
                            max_grade=9, grade_factor=LOW_GRADE_BOOST, seed=SEED)
print(f"\ntraining rows after weighting: {len(train_weighted)} "
      f"(x{1 + AUG_SHUFFLES} with shuffled copies = {len(train_weighted) * (1 + AUG_SHUFFLES)} examples per epoch)")
pd.DataFrame({"train": data["train"]["grade"].value_counts(), "weighted": train_weighted["grade"].value_counts()}).sort_index()

In [ ]:
import gc
import torch
from src.hf_model import load_model_and_tokenizer, load_with_dtype_check, ChoiceScorer, render_prompt
from src.prompting import build_messages, set_prompt_language

DTYPE = None

def fresh_model():
    """Base model without any adapter (each trial starts from scratch)."""
    global DTYPE, model
    try:
        del model
    except NameError:
        pass
    gc.collect(); torch.cuda.empty_cache()
    if FORCE_FP32:
        DTYPE = torch.float32
    if DTYPE is None:              # first load: pick bf16 / fp16 / fp32 for this GPU
        probe = [build_messages(r) for r in data["val"].head(8).to_dict(orient="records")]
        m, t, DTYPE = load_with_dtype_check(MODEL_ID, probe, load_in_4bit=LOAD_IN_4BIT)
    else:
        m, t = load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT, dtype=DTYPE)
    return m, t

model, tok = fresh_model()
print(f"dtype: {DTYPE}   GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
for lang in ("en", "si"):
    set_prompt_language(lang)
    print(f"\n--- prompt ({lang}) ---\n" + render_prompt(tok, build_messages(data["test"].iloc[0].to_dict())))

## 5. Baseline (untrained model)

In [ ]:
import json
from src.prompting import ShotSampler
from src.evaluate import run_eval, compute_metrics, save_metrics

def evaluate(model, run_name, splits=("val", "test"), lang="en"):
    set_prompt_language(lang)
    scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
    out = {}
    for split in splits:
        preds = run_eval(data[split], scorer, ShotSampler(data["train"], k=0),
                         RESULTS_DIR / run_name / f"{split}_predictions.jsonl", batch_size=64)
        m = compute_metrics(preds)
        save_metrics(m, RESULTS_DIR / run_name / f"{split}_metrics.json")
        out[split] = m["accuracy"]
        print(f"  {run_name} / {split}: {m['accuracy']:.2%}")
    return out

baseline = {lang: evaluate(model, f"base_{lang}", lang=lang) for lang in ("en", "si")} if DO_BASELINE else {}
pd.DataFrame(baseline).T.style.format("{:.2%}") if baseline else 'baseline skipped (DO_BASELINE = False)'

## 6. Hyperparameter search

Each trial trains a fresh adapter on a subset for one epoch and is scored on `val`. Finished trials are remembered (`hp_search.csv`), so rerunning skips them.

In [ ]:
import time
from src.hf_model import train_lora

search_file = RESULTS_DIR / "hp_search.csv"
done = pd.read_csv(search_file).to_dict(orient="records") if search_file.exists() else []
done_keys = {(d["lang"], d["lr"], d["r"]) for d in done}

if DO_SEARCH:
    subset = train_weighted.sample(n=min(SEARCH_TRAIN_SIZE, len(train_weighted)), random_state=SEED)
    for i, hp in enumerate(SEARCH_GRID):
        if (hp["lang"], hp["lr"], hp["r"]) in done_keys:
            continue
        if minutes_left() < 60:
            print(f"Skipping trial {hp}: less than an hour left in the time budget")
            continue
        print(f"\n=== trial {i + 1}/{len(SEARCH_GRID)}: {hp}")
        t0 = time.time()
        set_prompt_language(hp["lang"])
        model, tok = fresh_model()
        model, hist, best = train_lora(
            model, tok, subset, data["val"], RUNS_DIR / f"trial_{hp['lang']}_lr{hp['lr']}_r{hp['r']}",
            r=hp["r"], lr=hp["lr"], dropout=LORA_DROPOUT, epochs=1, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM,
            max_len=MAX_LEN, aug_shuffles=AUG_SHUFFLES, seed=SEED, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT,
            scorer_tokens=SCORER_TOKENS, max_minutes=minutes_left())
        minutes = (time.time() - t0) / 60
        done.append({**hp, "val_accuracy": best, "minutes": round(minutes, 1)})
        pd.DataFrame(done).to_csv(search_file, index=False)
        print(f"trial took {minutes:.0f} min -> val accuracy {best:.2%}")

search = pd.DataFrame(done).sort_values("val_accuracy", ascending=False) if done else pd.DataFrame()
BEST_HP = {k: search.iloc[0][k] for k in ("lang", "lr", "r")} if len(search) else DEFAULT_HP
BEST_HP = {"lang": str(BEST_HP["lang"]), "lr": float(BEST_HP["lr"]), "r": int(BEST_HP["r"])}
print("Best settings:", BEST_HP)
search

## 7. Final training (best settings, all training data)

In [ ]:
assert DO_FINAL, "DO_FINAL = False: stopping here (run the final training in a second run)"
t0 = time.time()
set_prompt_language(BEST_HP["lang"])
model, tok = fresh_model()
FINAL_DIR = RUNS_DIR / "final"
print(f"time budget for training: {minutes_left():.0f} min")
model, history, best_val = train_lora(
    model, tok, train_weighted, data["val"], FINAL_DIR, ckpt_dir=RUNS_DIR / "final_checkpoints",
    r=BEST_HP["r"], lr=BEST_HP["lr"], dropout=LORA_DROPOUT, epochs=EPOCHS, batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM, max_len=MAX_LEN, aug_shuffles=AUG_SHUFFLES, seed=SEED, dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT, scorer_tokens=SCORER_TOKENS, max_minutes=minutes_left())
json.dump({"model": MODEL_ID, **BEST_HP, "epochs": EPOCHS, "aug_shuffles": AUG_SHUFFLES,
           "low_grade_boost": LOW_GRADE_BOOST, "test_subject_boost": TEST_SUBJECT_BOOST, "best_val": best_val},
          open(FINAL_DIR / "settings.json", "w"), indent=2)
print(f"took {(time.time() - t0) / 60:.0f} min; best val accuracy {best_val:.2%}")
pd.DataFrame(history)

## 8. Test on the Dev Set

In [ ]:
final = evaluate(model, "final", lang=BEST_HP["lang"])
summary = pd.DataFrame({**{f"untrained ({k})": v for k, v in baseline.items()}, "fine-tuned": final}).T
summary.to_csv(RESULTS_DIR / "summary.csv")
summary.style.format("{:.2%}")

In [ ]:
def per(run, col):
    preds = pd.read_json(RESULTS_DIR / run / "test_predictions.jsonl", lines=True).merge(
        data["test"][["id", "grade"]], on="id")
    return preds.groupby(col)["correct"].mean()

base_run = f"base_{BEST_HP['lang']}"
has_base = (RESULTS_DIR / base_run / "test_predictions.jsonl").exists()
by_subject = pd.DataFrame({**({"untrained": per(base_run, "subject")} if has_base else {}),
                           "fine-tuned": per("final", "subject"), "questions": data["test"]["subject"].value_counts()})
if has_base:
    by_subject["gain"] = by_subject["fine-tuned"] - by_subject["untrained"]
by_subject.sort_values("fine-tuned", ascending=False).style.format(
    {"untrained": "{:.2%}", "fine-tuned": "{:.2%}", "gain": "{:+.2%}"}, na_rep="-")

In [ ]:
pd.DataFrame({**({"untrained": per(base_run, "grade")} if has_base else {}),
              "fine-tuned": per("final", "grade")}).style.format("{:.2%}")

In [ ]:
# Does the model read the questions, or favour one option number?
gold = data["test"]["answer"].value_counts(normalize=True).reindex([1, 2, 3, 4], fill_value=0)
pred = pd.read_json(RESULTS_DIR / "final" / "test_predictions.jsonl", lines=True)["pred"].value_counts(
    normalize=True).reindex([1, 2, 3, 4], fill_value=0)
pd.DataFrame({"gold": gold, "fine-tuned picks": pred}).style.format("{:.1%}")

## 9. Files

* Adapter (with `settings.json`): `outputs/gemma3-4b-sinhala/final/` in `PERSIST_DIR` — on Kaggle, the version's **Output** tab.
* Results: `results/gemma3-4b-sinhala/` — `hp_search.csv`, `summary.csv`, predictions and metrics per run.

To use the model later:

```python
from peft import PeftModel
from src.prompting import set_prompt_language
model, tok = load_model_and_tokenizer(MODEL_ID, load_in_4bit=True)
model = PeftModel.from_pretrained(model, "outputs/gemma3-4b-sinhala/final")
set_prompt_language(json.load(open("outputs/gemma3-4b-sinhala/final/settings.json"))["lang"])
scorer = ChoiceScorer(model, tok)
```